# **Streaming Taxi Ride Locations to a Bubble Map**

## **Objective**
The goal of this project is to build a small-scale streaming pipeline that reads real-time taxi ride data, extracts key geographic fields (latitude and longitude), and visualizes the locations on an interactive map using bubble markers.  
This helps to better understand where taxi rides are clustered across New York City.

---

## **Steps Taken**

### 1. **Environment Setup**
- Created and selected a Google Cloud project.
- Enabled required APIs: Dataflow API, BigQuery API, Pub/Sub API, Cloud Storage API.
- Created a new Cloud Storage bucket for temporary files.

### 2. **Dataset and Table Creation**
- Created a BigQuery dataset named `taxirides`.
- Created a table called `realtime` inside `taxirides`.
- Defined the schema with important fields like `ride_id`, `latitude`, `longitude`, `timestamp`, etc.

### 3. **Streaming Pipeline Development**
- Cloned and modified a sample Apache Beam Python pipeline.
- The pipeline reads streaming messages from Pub/Sub.
- It processes the incoming messages, windows them by publish time, and writes them into Google Cloud Storage.

### 4. **Data Extraction**
- Queried the BigQuery `taxirides.realtime` table to select valid (non-null) latitude and longitude values.
- Retrieved a subset of **500 rows** for visualization purposes.

```sql
SELECT Latitude, Longitude
FROM `big-data-platforms-44.taxirides.realtime`
WHERE Latitude IS NOT NULL AND Longitude IS NOT NULL
LIMIT 1000;
```

I exported data as CSV from Google Sheets which has the first 500 rows

In [9]:
import pandas as pd

# Load both CSVs
taxi_rides = pd.read_csv("taxi_rides.csv")

# Preview the result
taxi_rides.head()


,Latitude,Longitude
0,40.73254,-73.97445
1,40.73026,-73.95130
2,40.63230,-73.96674
3,40.72230,-73.99710
4,40.72033,-74.01228


In [10]:
import folium

# Center the map roughly around NYC (adjust if needed)
map_center = [40.730610, -73.935242]
taxi_map = folium.Map(location=map_center, zoom_start=11)

# Add each ride as a blue bubble
for _, row in taxi_rides.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,                  # Slightly bigger bubbles if you like
        color='#4285F4',            # Google's nice blue (#4285F4)
        fill=True,
        fill_color='#4285F4',
        fill_opacity=0.6
    ).add_to(taxi_map)

taxi_map


## **Results**
- Successfully plotted 500 real taxi rides over the New York City area.
- Major clusters were visible around Manhattan, Downtown Brooklyn, and parts of Queens.
- Bubble size and density clearly indicated hotspots of taxi activity.

---

## **Conclusion**
This project demonstrated a basic but powerful real-time data pipeline from Pub/Sub to BigQuery and finally to a visual map.  
It showed that simple geographic data can tell strong stories about human activity patterns.  
Future work could involve:
- Plotting full real-time streams instead of samples.
- Adding pop-up tooltips to bubbles for richer insights (e.g., timestamp, ride_id).
- Using clustering techniques to handle larger datasets for better performance.